# Streaming

<img src="./assets/LC_streaming.png" width="400">

Streaming reduces the latency between generating data and the user receiving it.
There are two types frequently used with Agents:

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

CRUSOE_API_KEY=****bmsK
LANGSMITH_API_KEY=****here
LANGSMITH_TRACING=false
LANGSMITH_PROJECT=****ials


In [2]:
from langchain.agents import create_agent

In [3]:
from langchain_crusoe import ChatCrusoe
agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    system_prompt="You are a full-stack comedian",
)

## No Streaming (invoke)

In [4]:
result = agent.invoke({"messages": [{"role": "user", "content": "Tell me a joke"}]})
print(result["messages"][1].content)

I’ve been doing full-stack development for years. You know what a full-stack developer is, right? It’s basically just a person who has the unique ability to completely ruin both the user interface *and* the database architecture in a single Git commit. 

But anyway, here's a joke for you:

Why did the full-stack developer get dumped by their partner?

Because the communication was asynchronous, the frontend was completely unresponsive, and every time they had a serious argument, they tried to `DROP TABLE` and flee the relationship. 

*Ba-dum tss!* I'll be here all week. Try the API, and don't forget to tip your server. (Especially if it's running Node.js, that thing needs all the help it can get.)


## values
You have seen this streaming mode in our examples so far. 

In [5]:
# Stream = values
for step in agent.stream(
    {"messages": [{"role": "user", "content": "Tell me a Dad joke"}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Tell me a Dad joke


================================== Ai Message ==================================

A SQL query walks into a bar, walks up to two tables, and asks... 

"Can I JOIN you?"

...And since I'm a *full-stack* comedian, I've got one for the frontend too:

Why did the CSS developer go to therapy?

Because he had too many issues with his parent elements and couldn't center his inner child! 

*Ba-dum tss!* 🥁


## messages
Messages stream data token by token - the lowest latency possible. This is perfect for interactive applications like chatbots.

In [6]:
for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "Write me a family friendly poem."}]},
    stream_mode="messages",
):
    print(f"{token.content}", end="")

I tossed them in the washing machine,
The brightest socks you've ever seen.
A perfect pair, a matched-up set,
The coolest socks a kid could get.

I poured the soap and shut the door,
Then heard a loud

 and rhythmic roar.
The water spun, the clothes went round,
And tumbled safely to the ground.

But when I pulled the wet clothes out,
I gave a sad and startled

 shout!
I found a shirt, a pair of jeans,
A towel that smelled of tangerines—

But one striped sock had disappeared!
I checked the tub, I peaked and pe

ered.
I asked my dad, "Where did it go?"
He said, "Ask the dog, he ought to know."

I asked the dog, he barked and ran,
And hid his face inside a pan.
I think he knows the secret truth,
But he

's not talking—missing tooth.

I’ve come to learn, through laundry woe,
That washers have a place to go.
A hidden portal, a secret gate

,
Where lonely socks await their fate.

I think there is a Laundry Beast,
Who hosts a mismatched, sock-y feast.
He takes the left, he leaves the right,
And wears them on his toes at night.

So if you’re missing one

 blue sock,
Or wonder where your polka-dots walk,
Don’t blame the dog, don’t blame the cat,
The Laundry Beast has stolen that!

## Tools can stream too!
Streaming generally means delivering information to the user before the final result is ready. There are many cases where this is useful. A `get_stream_writer` writer allows you to easily stream `custom` data from sources you create.

In [7]:
from langchain_crusoe import ChatCrusoe
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"


agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    print(chunk)

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='08fc5d4c-9c6a-4608-9970-ca0b1728b2b6')]})


('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='08fc5d4c-9c6a-4608-9970-ca0b1728b2b6'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 161, 'total_tokens': 191, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 17, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 64, 'multimodal_tokens': None}}, 'model_provider': 'openai', 'model_name': 'zai/GLM-5.2', 'system_fingerprint': None, 'id': 'chatcmpl-___prefill_addr_10.234.11.50:8998___decode_addr_10.234.34.214:8998_579de60c045444a1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03964-e209-7162-9974-80f874a2f78c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'chatcmpl-tool-bed73b91a6830b39'

('values', {'messages': [HumanMessage(content='What is the weather in SF?', additional_kwargs={}, response_metadata={}, id='08fc5d4c-9c6a-4608-9970-ca0b1728b2b6'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 161, 'total_tokens': 191, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 17, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 64, 'multimodal_tokens': None}}, 'model_provider': 'openai', 'model_name': 'zai/GLM-5.2', 'system_fingerprint': None, 'id': 'chatcmpl-___prefill_addr_10.234.11.50:8998___decode_addr_10.234.34.214:8998_579de60c045444a1', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a03964-e209-7162-9974-80f874a2f78c-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'chatcmpl-tool-bed73b91a6830b39'

In [8]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["custom"],
):
    print(chunk)

('custom', 'Looking up data for city: San Francisco')
('custom', 'Acquired data for city: San Francisco')


## Try different modes on your own!
Modify the stream mode and the select to produce different results.

In [9]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["values", "custom"],
):
    if chunk[0] == "custom":
        print(chunk[1])

Looking up data for city: San Francisco
Acquired data for city: San Francisco
